In [1]:
import oceanbench

oceanbench.__version__

'0.5.1'

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [2]:
# glowcascade_final over the full official start set, cut to NINE lead days.
#
# 52 Wednesday challenger folders of 2024, 20240103 through 20241225, each
# initialised from the as-issued GLO12 nowcast of the Tuesday before it and
# forced by the IFS forecast issued that same Tuesday.
#
# WHY NINE. The IFS forecast package carries lead_day_index 0..9, that is the
# forcing of forecast days 1..10 minus its last day, so the tenth forecast day
# is driven by PERSISTED lead 9 forcing rather than by a forecast. Julien's
# decision of 2026-09-10: the entry scores the nine days that are forced by a
# real IFS forecast and stops there. Nothing is recomputed and no forecast
# zarr is touched: the store still holds ten days per start, this module drops
# the last time step on the way in.
#
# This copy is scored under oceanbench origin/main 7e5ec87 (pending 0.6.0).
import datetime
import pathlib

import xarray

_ROOT = pathlib.Path("/mnt/data/glonet2/ifs21/forecasts/glowcascade117m_v5_ring")
_PATHS = sorted(_ROOT.glob("2024*.zarr"))
_FIRST_DAYS = [datetime.datetime.strptime(p.stem, "%Y%m%d") for p in _PATHS]
_LEAD_DAYS = 9


def _prepared(dataset: xarray.Dataset) -> xarray.Dataset:
    dataset = dataset.isel(time=slice(0, _LEAD_DAYS))
    lead_count = dataset.sizes["time"]
    return dataset.rename({"time": "lead_day_index"}).assign_coords({"lead_day_index": range(lead_count)})


challenger_dataset: xarray.Dataset = xarray.open_mfdataset(
    [str(p) for p in _PATHS],
    engine="zarr",
    preprocess=_prepared,
    combine="nested",
    concat_dim="first_day_datetime",
    parallel=False,
).assign_coords({"first_day_datetime": _FIRST_DAYS})


### Evaluation configuration

In [3]:
region = 'global'

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to GLORYS reanalysis

In [4]:
oceanbench.metrics.rmsd_of_variables_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.066707,0.067033,0.067066,0.067175,0.067572,0.068209,0.069149,0.070386,0.071382
Temperature (°C) [sea_water_potential_temperature]{surface},0.516623,0.516809,0.516979,0.519226,0.523765,0.531615,0.542516,0.554669,0.564486
Salinity (PSU) [sea_water_salinity]{surface},0.571422,0.566964,0.562602,0.558788,0.554864,0.551458,0.548657,0.546033,0.542754
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.118482,0.118812,0.119459,0.120662,0.122343,0.124674,0.127613,0.130929,0.133505
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.119438,0.119824,0.120590,0.121906,0.123931,0.126576,0.129717,0.133113,0.135728
Temperature (°C) [sea_water_potential_temperature]{50m},0.864215,0.863400,0.861716,0.860298,0.860634,0.862363,0.865620,0.870011,0.872078
Salinity (PSU) [sea_water_salinity]{50m},0.242325,0.242057,0.241659,0.241306,0.241125,0.241130,0.241273,0.241459,0.241351
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.112210,0.112500,0.112746,0.113240,0.114028,0.115230,0.116862,0.118768,0.119991
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.113204,0.113193,0.113314,0.113703,0.114363,0.115398,0.116952,0.118727,0.119895
Temperature (°C) [sea_water_potential_temperature]{100m},1.058949,1.058840,1.058332,1.058867,1.060494,1.063965,1.069689,1.075236,1.077739


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLORYS reanalysis

In [5]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},40.96176,41.267026,41.590627,41.904512,42.25835,42.658058,43.141259,43.637555,43.940872


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLORYS reanalysis

In [6]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.116883,0.116367,0.116145,0.116632,0.117751,0.119046,0.121291,0.123567,0.125178
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.122925,0.121775,0.121515,0.121589,0.122646,0.123926,0.126522,0.129079,0.131272


#### Root Mean Square Deviation (RMSD) of variables compared to observations

In [7]:
oceanbench.metrics.rmsd_of_variables_compared_to_observations(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Observations
Temperature (°C) [sea_water_potential_temperature]{surface},0.735546,0.759160,0.728743,0.752879,0.776035,0.816609,0.814756,0.830192,0.852788,150475
Temperature (°C) [sea_water_potential_temperature]{0-5m},0.725528,0.738120,0.759506,0.784830,0.786163,0.796604,0.808180,0.806948,0.804174,80886
Temperature (°C) [sea_water_potential_temperature]{5-100m},0.861734,0.871038,0.860048,0.888018,0.879697,0.882238,0.900872,0.933208,0.932163,1341567
Temperature (°C) [sea_water_potential_temperature]{100-300m},0.781218,0.802540,0.793219,0.796668,0.810656,0.811004,0.833298,0.835828,0.865748,2132232
Temperature (°C) [sea_water_potential_temperature]{300-600m},0.519773,0.532026,0.528961,0.529362,0.538315,0.555807,0.555928,0.570481,0.584227,2646322
Salinity (PSU) [sea_water_salinity]{0-5m},0.250842,0.277421,0.268459,0.302216,0.272403,0.278264,0.268386,0.260965,0.291752,69024
Salinity (PSU) [sea_water_salinity]{5-100m},0.195085,0.199990,0.204952,0.204425,0.204906,0.205666,0.207378,0.205419,0.209787,1143321
Salinity (PSU) [sea_water_salinity]{100-300m},0.126623,0.128597,0.129267,0.128910,0.131872,0.130698,0.136337,0.133195,0.135957,1816368
Salinity (PSU) [sea_water_salinity]{300-600m},0.080315,0.080603,0.080207,0.080513,0.081292,0.083174,0.082929,0.086520,0.087100,2245406
Sea level anomaly (m) [sea_surface_height_above_geoid]{surface},0.048044,0.048958,0.049700,0.051098,0.052333,0.054134,0.055691,0.057387,0.059433,15465310


#### Deviation of Lagrangian trajectories compared to GLORYS reanalysis

In [8]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8
Lagrangian trajectory deviation (km) []{surface},9.954945,19.216137,28.0968,36.707722,45.118183,53.360073,61.45237


#### Root Mean Square Deviation (RMSD) of variables compared to GLO12 analysis

In [9]:
oceanbench.metrics.rmsd_of_variables_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.013663,0.016594,0.019148,0.022447,0.026357,0.030419,0.034746,0.039100,0.042224
Temperature (°C) [sea_water_potential_temperature]{surface},0.172826,0.200357,0.228998,0.258718,0.290088,0.322971,0.358040,0.392085,0.417672
Salinity (PSU) [sea_water_salinity]{surface},0.127692,0.146971,0.163078,0.177607,0.191124,0.206260,0.221697,0.236023,0.247604
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.049199,0.055950,0.062801,0.070519,0.078776,0.087177,0.096062,0.104381,0.110530
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.050934,0.057994,0.065081,0.072660,0.080976,0.089680,0.098468,0.106951,0.113204
Temperature (°C) [sea_water_potential_temperature]{50m},0.304798,0.331677,0.358790,0.388576,0.421670,0.458555,0.497412,0.532897,0.555082
Salinity (PSU) [sea_water_salinity]{50m},0.063009,0.067996,0.073616,0.079766,0.086528,0.093830,0.101405,0.108469,0.113273
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.044125,0.048678,0.053916,0.059668,0.066145,0.073111,0.080523,0.087531,0.092348
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.046161,0.050677,0.055750,0.061444,0.067737,0.074536,0.081721,0.088707,0.093691
Temperature (°C) [sea_water_potential_temperature]{100m},0.274855,0.309140,0.344564,0.384617,0.427515,0.473170,0.520700,0.563949,0.590980


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLO12 analysis

In [10]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},30.944076,31.960049,32.828756,33.730414,34.647253,35.618845,36.611095,37.485835,38.183533


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLO12 analysis

In [11]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.045908,0.055182,0.062518,0.069371,0.076275,0.083578,0.091011,0.097956,0.102881
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.048264,0.059477,0.068025,0.075421,0.082650,0.090291,0.097952,0.105302,0.110112


#### Deviation of Lagrangian trajectories compared to GLO12 analysis

In [12]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8
Lagrangian trajectory deviation (km) []{surface},3.888134,7.70474,11.747609,16.170723,21.039541,26.384569,32.187279
